In [1]:
import pandas as pd

In [2]:
lead_log = pd.read_csv('lead_log.csv')

In [3]:
user_referrals = pd.read_csv('user_referrals.csv')

In [4]:
paid_transactions = pd.read_csv('paid_transactions.csv')

In [5]:
user_logs = pd.read_csv('user_logs.csv')

In [6]:
user_referral_logs = pd.read_csv('user_referral_logs.csv')

In [7]:
user_referral_statuses = pd.read_csv('user_referral_statuses.csv')

In [8]:
referral_rewards = pd.read_csv('referral_rewards.csv')

In [9]:

all_tables = {
    "Lead Log": lead_log,
    "User Referrals": user_referrals,
    "Paid Transactions": paid_transactions,
    "User Logs": user_logs,
    "User Referral Logs": user_referral_logs,
    "User Referral Statuses": user_referral_statuses,
    "Referral Rewards": referral_rewards
}

for name, df in all_tables.items():
    print(f"--- Profiling Report for: {name} ---")
    null_counts = df.isnull().sum()
    distinct_counts = df.nunique()
    
    
    profile_df = pd.DataFrame({'Null Count': null_counts, 'Distinct Count': distinct_counts})
    print(profile_df)
    print("\n" + "="*40 + "\n")

--- Profiling Report for: Lead Log ---
                    Null Count  Distinct Count
id                           0               8
lead_id                      0               5
source_category              0               2
created_at                   0               5
preferred_location           0               3
timezone_location            0               1
current_status               0               5


--- Profiling Report for: User Referrals ---
                         Null Count  Distinct Count
referral_at                       0              44
referral_id                       0              46
referee_id                        3              37
referee_name                      3              35
referee_phone                     0              33
referral_reward_id               38               3
referral_source                   0               3
referrer_id                      13              13
transaction_id                   13              27
updated_at        

In [10]:

lead_log['current_status'] = lead_log['current_status'].str.capitalize()
user_logs['name'] = user_logs['name'].str.capitalize()


lead_log = lead_log.dropna(subset=['lead_id'])
user_referrals = user_referrals.dropna(subset=['referral_id'])

print("Data Cleaning and Capitalization Complete.")

Data Cleaning and Capitalization Complete.


In [12]:

lead_log['created_at'] = pd.to_datetime(lead_log['created_at'])
paid_transactions['transaction_at'] = pd.to_datetime(paid_transactions['transaction_at'])
user_referrals['updated_at'] = pd.to_datetime(user_referrals['updated_at'])

lead_log['local_created_at'] = lead_log['created_at'].dt.tz_convert('Asia/Jakarta')
paid_transactions['local_transaction_at'] = paid_transactions['transaction_at'].dt.tz_convert('Asia/Jakarta')

print("Time Adjustment Complete.")

Time Adjustment Complete.


In [13]:

merged_df = pd.merge(user_referrals, lead_log, left_on='referee_id', right_on='lead_id', how='left')
final_pipeline = pd.merge(merged_df, paid_transactions, on='transaction_id', how='left')

print(f"Merge Complete. Final row count: {len(final_pipeline)}")

Merge Complete. Final row count: 58


In [14]:

final_report = final_pipeline.drop_duplicates(subset=['referral_id']).head(46)


def validate_referral(row):
    
    if pd.isna(row['transaction_id']):
        return False
    
   
    if row['transaction_status'] != 'PAID':
        return False
        
    return True


final_report['is_business_logic_valid'] = final_report.apply(validate_referral, axis=1)

print(f"Business Logic Applied. Final row count: {len(final_report)}")
final_report[['referral_id', 'is_business_logic_valid']].head()

Business Logic Applied. Final row count: 46


,referral_id,is_business_logic_valid
0,9331c8f144dad5a3b8e4a10467b4343a,False
1,6371079a92bcbf0c16ae5fcdf4fc9c10,False
2,a49105b02e690472452527663559d97a,True
3,fcb804c8ff24e5b2974a7e965ebea5e8,False
4,9e9324e6fde29bb0d230654b38ccfdd4,True


In [15]:

final_report.to_csv('final_assessment_report.csv', index=False)
print("Assessment Complete! 'final_assessment_report.csv' has been saved to your folder.")

Assessment Complete! 'final_assessment_report.csv' has been saved to your folder.
